# Re-download Daily Prices v2 — OHLCV (no TR fields)

**What changed from v1:**
- Uses `rd.get_history()` **without `fields`** — returns default OHLCV from the timeseries endpoint
- v1 used `TR.PriceClose` which is a fundamental/reference field that returns **single snapshots** for many stocks instead of daily time series
- This is why 43 stocks only had 1 price observation despite being active, liquid stocks (DLTA, HERO, SCCO, LPPF, TOTO, etc.)

**Settings:** batch size 10, 3s sleep, 3 retries, 5 chunks (2012–2025)

**Run order:** Cell 1 → 2 → 3 → 4 (skip 5 unless you want the probe) → 6 → 7 → 8 → 9

In [21]:
import refinitiv.data as rd
import pandas as pd
import numpy as np
from datetime import datetime
import time
import json
import os
import warnings
warnings.filterwarnings('ignore')

rd.open_session()

<refinitiv.data.session.Definition object at 0x10a772420 {name='workspace'}>

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────
DAILY_BATCH   = 10
SLEEP_BETWEEN = 1
MAX_RETRIES   = 3

CHUNKS = [
    (datetime(2008, 1, 1),  datetime(2009, 12, 31)),
    (datetime(2009, 1, 1),  datetime(2011, 12, 31)),
    (datetime(2012, 1, 1),  datetime(2014, 12, 31)),
    (datetime(2015, 1, 1),  datetime(2017, 12, 31)),
    (datetime(2018, 1, 1),  datetime(2020, 12, 31)),
    (datetime(2021, 1, 1),  datetime(2023, 12, 31)),
    (datetime(2024, 1, 1),  datetime(2025, 12, 31)),
]

# Chunk CSV prefix — v2 to avoid overwriting v1 files
CHUNK_PREFIX = 'v2_daily_chunk'

all_rics = pd.read_csv('idx_ric_list.csv')['RIC'].tolist()
download_rics = all_rics

print(f'RICs to download: {len(download_rics)}')
print(f'Batches per chunk: {(len(download_rics) + DAILY_BATCH - 1) // DAILY_BATCH}')
print(f'Total batches: {len(CHUNKS) * ((len(download_rics) + DAILY_BATCH - 1) // DAILY_BATCH)}')

RICs to download: 992
Batches per chunk: 100
Total batches: 500


In [23]:
def fetch_batch(rics, start, end, retries=MAX_RETRIES):
    """Download daily OHLCV for a batch of RICs with retries.
    
    KEY CHANGE: no 'fields' parameter — uses default timeseries endpoint
    which returns OPEN, HIGH, LOW, CLOSE, VOLUME, COUNT.
    """
    for attempt in range(1, retries + 1):
        try:
            df = rd.get_history(
                universe=rics,
                interval='1D',
                start=start.strftime('%Y-%m-%d'),
                end=end.strftime('%Y-%m-%d')
            )
            if df is not None and not df.empty:
                return df, None
            else:
                return None, None
        except Exception as e:
            err = str(e)[:100]
            if attempt < retries:
                wait = SLEEP_BETWEEN * (2 ** attempt)
                time.sleep(wait)
            else:
                return None, err
    return None, 'max retries exceeded'

## (Optional) Quick probe — skip to go straight to download

Tests 10 RICs for 1 month to verify the API returns multi-column OHLCV data.

In [24]:
# Quick sanity check — run this to confirm OHLCV columns come back
test_rics = ['AALI.JK', 'BBCA.JK', 'TLKM.JK', 'ASII.JK', 'BMRI.JK',
             'DLTA.JK', 'HERO.JK', 'SCCO.JK', 'LPPF.JK', 'TOTO.JK']
test_df, test_err = fetch_batch(test_rics, datetime(2024, 1, 1), datetime(2024, 1, 31))

if test_df is not None:
    print(f'Shape: {test_df.shape}')
    print(f'Columns: {list(test_df.columns)}')
    print(f'\nFirst 5 rows:')
    print(test_df.head())
else:
    print(f'FAILED: {test_err}')

Shape: (22, 140)
Columns: [('AALI.JK', 'TRDPRC_1'), ('AALI.JK', 'HIGH_1'), ('AALI.JK', 'LOW_1'), ('AALI.JK', 'ACVOL_UNS'), ('AALI.JK', 'OPEN_PRC'), ('AALI.JK', 'BID'), ('AALI.JK', 'ASK'), ('AALI.JK', 'TRNOVR_UNS'), ('AALI.JK', 'VWAP'), ('AALI.JK', 'NUM_MOVES'), ('AALI.JK', 'FRGN_BVAL'), ('AALI.JK', 'FRGN_SVAL'), ('AALI.JK', 'VWAP_VOL'), ('AALI.JK', 'NAVALUE'), ('BBCA.JK', 'TRDPRC_1'), ('BBCA.JK', 'HIGH_1'), ('BBCA.JK', 'LOW_1'), ('BBCA.JK', 'ACVOL_UNS'), ('BBCA.JK', 'OPEN_PRC'), ('BBCA.JK', 'BID'), ('BBCA.JK', 'ASK'), ('BBCA.JK', 'TRNOVR_UNS'), ('BBCA.JK', 'VWAP'), ('BBCA.JK', 'NUM_MOVES'), ('BBCA.JK', 'FRGN_BVAL'), ('BBCA.JK', 'FRGN_SVAL'), ('BBCA.JK', 'VWAP_VOL'), ('BBCA.JK', 'NAVALUE'), ('TLKM.JK', 'TRDPRC_1'), ('TLKM.JK', 'HIGH_1'), ('TLKM.JK', 'LOW_1'), ('TLKM.JK', 'ACVOL_UNS'), ('TLKM.JK', 'OPEN_PRC'), ('TLKM.JK', 'BID'), ('TLKM.JK', 'ASK'), ('TLKM.JK', 'TRNOVR_UNS'), ('TLKM.JK', 'VWAP'), ('TLKM.JK', 'NUM_MOVES'), ('TLKM.JK', 'FRGN_BVAL'), ('TLKM.JK', 'FRGN_SVAL'), ('TLKM.JK', 'V

## Download all chunks

Each chunk saves to `v2_daily_chunk_N.csv`. If a chunk file already exists, it is skipped.

**OHLCV stacking:** `get_history()` without fields returns a DataFrame with MultiIndex columns `(field, instrument)`. After stacking, we keep CLOSE and VOLUME.

In [25]:
for chunk_idx, (chunk_start, chunk_end) in enumerate(CHUNKS):
    chunk_file = f'{CHUNK_PREFIX}_{chunk_idx}.csv'
    
    if os.path.exists(chunk_file):
        existing = pd.read_csv(chunk_file)
        print(f'\nChunk {chunk_idx} ({chunk_start.date()} to {chunk_end.date()}): '
              f'ALREADY DONE — {len(existing):,} rows, '
              f'{existing["Instrument"].nunique()} stocks')
        continue
    
    print(f'\n{"="*65}')
    print(f'  Chunk {chunk_idx}: {chunk_start.date()} to {chunk_end.date()}')
    print(f'{"="*65}')
    
    frames = []
    permanently_failed = []
    total_batches = (len(download_rics) + DAILY_BATCH - 1) // DAILY_BATCH
    
    for i in range(0, len(download_rics), DAILY_BATCH):
        batch_rics = download_rics[i:i + DAILY_BATCH]
        batch_num = i // DAILY_BATCH + 1
        
        df, err = fetch_batch(batch_rics, chunk_start, chunk_end)
        
        if df is not None:
            # Stack each batch to long format IMMEDIATELY (before concat)
            # MultiIndex columns are (Instrument, Field)
            if isinstance(df.columns, pd.MultiIndex):
                long = df.stack(level=0).reset_index()
                long = long.rename(columns={'level_1': 'Instrument'})
                long = long.rename(columns={
                    'TRDPRC_1': 'Price',
                    'ACVOL_UNS': 'Volume'
                })
                long = long[['Date', 'Instrument', 'Price', 'Volume']]
            else:
                long = df.reset_index()
                long = long.rename(columns={
                    'TRDPRC_1': 'Price',
                    'ACVOL_UNS': 'Volume'
                })
            frames.append(long)
        elif err:
            permanently_failed.append({
                'rics': batch_rics, 'error': err
            })
        
        if batch_num % 10 == 0:
            print(f'  Batch {batch_num}/{total_batches} — '
                  f'{len(frames)} successful, {len(permanently_failed)} failed')
        
        time.sleep(SLEEP_BETWEEN)
    
    if frames:
        chunk_long = pd.concat(frames, ignore_index=True)
        chunk_long['Price'] = pd.to_numeric(chunk_long['Price'], errors='coerce')
        chunk_long['Volume'] = pd.to_numeric(chunk_long['Volume'], errors='coerce')
        
        # Drop rows where both Price and Volume are NaN
        chunk_long = chunk_long.dropna(subset=['Price', 'Volume'], how='all')
        # Dedup on (Date, Instrument) — NOT on date index alone
        chunk_long = chunk_long.drop_duplicates(subset=['Date', 'Instrument'], keep='first')
        
        chunk_long.to_csv(chunk_file, index=False)
        print(f'\n  Saved {chunk_file}: {len(chunk_long):,} rows, '
              f'{chunk_long["Instrument"].nunique()} stocks, '
              f'{chunk_long["Price"].notna().mean()*100:.1f}% price coverage')
    else:
        print(f'\n  No data for chunk {chunk_idx}!')
    
    if permanently_failed:
        print(f'  Permanently failed: {len(permanently_failed)} batches')
        fail_file = f'{CHUNK_PREFIX}_{chunk_idx}_failed.json'
        with open(fail_file, 'w') as f:
            json.dump(permanently_failed, f, indent=2)
        print(f'  Failed RICs saved to {fail_file}')

print(f'\n{"="*65}')
print('  ALL CHUNKS COMPLETE')
print(f'{"="*65}')


  Chunk 0: 2012-01-01 to 2014-12-31
  Batch 10/100 — 10 successful, 0 failed
  Batch 20/100 — 20 successful, 0 failed
  Batch 30/100 — 30 successful, 0 failed
  Batch 40/100 — 40 successful, 0 failed
  Batch 50/100 — 50 successful, 0 failed
  Batch 60/100 — 60 successful, 0 failed
  Batch 70/100 — 70 successful, 0 failed
  Batch 80/100 — 80 successful, 0 failed
  Batch 90/100 — 90 successful, 0 failed
  Batch 100/100 — 99 successful, 0 failed

  Saved v2_daily_chunk_0.csv: 257,420 rows, 510 stocks, 100.0% price coverage

  Chunk 1: 2015-01-01 to 2017-12-31
  Batch 10/100 — 10 successful, 0 failed
  Batch 20/100 — 20 successful, 0 failed
  Batch 30/100 — 30 successful, 0 failed
  Batch 40/100 — 40 successful, 0 failed
  Batch 50/100 — 50 successful, 0 failed
  Batch 60/100 — 60 successful, 0 failed
  Batch 70/100 — 70 successful, 0 failed
  Batch 80/100 — 80 successful, 0 failed
  Batch 90/100 — 90 successful, 0 failed
  Batch 100/100 — 99 successful, 0 failed

  Saved v2_daily_chunk_1

## Combine all chunks

In [26]:
all_chunks = []
for chunk_idx in range(len(CHUNKS)):
    chunk_file = f'{CHUNK_PREFIX}_{chunk_idx}.csv'
    if os.path.exists(chunk_file):
        df = pd.read_csv(chunk_file, parse_dates=['Date'])
        all_chunks.append(df)
        print(f'  Loaded {chunk_file}: {len(df):,} rows, {df["Instrument"].nunique()} stocks')
    else:
        print(f'  MISSING: {chunk_file}')

daily_long = pd.concat(all_chunks, ignore_index=True)
daily_long = daily_long.drop_duplicates(subset=['Date', 'Instrument'], keep='first')
daily_long = daily_long.sort_values(['Instrument', 'Date']).reset_index(drop=True)

print(f'\nCombined daily panel: {len(daily_long):,} rows, '
      f'{daily_long["Instrument"].nunique()} stocks')
print(f'Date range: {daily_long["Date"].min().date()} to {daily_long["Date"].max().date()}')
print(f'Price coverage: {daily_long["Price"].notna().mean()*100:.1f}%')

daily_long.to_csv('idx_daily_prices.csv', index=False)
print('\nSaved idx_daily_prices.csv')

  Loaded v2_daily_chunk_0.csv: 257,420 rows, 510 stocks
  Loaded v2_daily_chunk_1.csv: 289,215 rows, 569 stocks
  Loaded v2_daily_chunk_2.csv: 390,618 rows, 718 stocks
  Loaded v2_daily_chunk_3.csv: 512,442 rows, 839 stocks
  Loaded v2_daily_chunk_4.csv: 377,679 rows, 881 stocks

Combined daily panel: 1,827,374 rows, 967 stocks
Date range: 2012-01-02 to 2025-12-30
Price coverage: 100.0%

Saved idx_daily_prices.csv


## Rebuild master panel

In [27]:
jci = pd.read_csv('jci_index.csv', parse_dates=['Date'])
rf  = pd.read_csv('bi_riskfree_rate.csv', parse_dates=['Date'])

master = pd.merge(daily_long, jci[['Date', 'Market_Return']], on='Date', how='left')
master = pd.merge(master, rf[['Date', 'Daily_Rf']], on='Date', how='left')

master = master.sort_values(['Instrument', 'Date'])
master['Stock_Return'] = master.groupby('Instrument')['Price'].transform(
    lambda x: np.log(x / x.shift(1))
)
master['Excess_Return'] = master['Stock_Return'] - master['Daily_Rf']
master['Market_Excess'] = master['Market_Return'] - master['Daily_Rf']
master['DayOfWeek'] = master['Date'].dt.dayofweek
master['DayName']   = master['Date'].dt.day_name()
master['YearMonth'] = master['Date'].dt.to_period('M')

master.to_csv('idx_master_panel.csv', index=False)

valid = master.dropna(subset=['Stock_Return'])
print(f'Master panel: {len(master):,} rows, {master["Instrument"].nunique()} stocks')
print(f'Valid stock returns: {len(valid):,} ({len(valid)/len(master)*100:.1f}%)')
print(f'\nSaved idx_master_panel.csv')

Master panel: 1,827,374 rows, 967 stocks
Valid stock returns: 1,826,405 (99.9%)

Saved idx_master_panel.csv


## Quality check

In [28]:
valid = master.dropna(subset=['Stock_Return']).copy()
valid['Year'] = valid['Date'].dt.year

print('Stocks with valid returns by year:')
print(valid.groupby('Year')['Instrument'].nunique().to_string())

pre  = valid[(valid['Date'] >= '2012-01-01') & (valid['Date'] <= '2016-12-31')]
post = valid[(valid['Date'] >= '2017-01-01') & (valid['Date'] <= '2024-12-31')]
print(f'\nPre  (2012-2016): {len(pre):,} obs, {pre["Instrument"].nunique()} stocks')
print(f'Post (2017-2024): {len(post):,} obs, {post["Instrument"].nunique()} stocks')

both = set(pre['Instrument'].unique()) & set(post['Instrument'].unique())
print(f'In both periods: {len(both)} stocks')

# Compare with v1
print(f'\n--- Comparison with v1 (TR.PriceClose) ---')
print(f'v1 had: 110 stocks, 36 with >500 prices, 43 single-snapshot')
print(f'v2 has: {daily_long["Instrument"].nunique()} stocks, '
      f'{(daily_long.groupby("Instrument")["Price"].apply(lambda x: x.notna().sum()) > 500).sum()} with >500 prices')

Stocks with valid returns by year:
Year
2012    445
2013    479
2014    491
2015    511
2016    519
2017    556
2018    606
2019    658
2020    695
2021    720
2022    759
2023    823
2024    854
2025    861

Pre  (2012-2016): 441,671 obs, 544 stocks
Post (2017-2024): 1,194,177 obs, 921 stocks
In both periods: 525 stocks

--- Comparison with v1 (TR.PriceClose) ---
v1 had: 110 stocks, 36 with >500 prices, 43 single-snapshot
v2 has: 967 stocks, 873 with >500 prices


## Retry failures

In [29]:
import glob as globmod

fail_files = sorted(globmod.glob(f'{CHUNK_PREFIX}_*_failed.json'))
if not fail_files:
    print('No failed batches to retry!')
else:
    for ff in fail_files:
        parts = ff.replace('.json', '').split('_')
        chunk_idx = int(parts[-2])
        chunk_start, chunk_end = CHUNKS[chunk_idx]
        
        with open(ff) as f:
            failures = json.load(f)
        
        print(f'\nRetrying {len(failures)} failed batches from chunk {chunk_idx}...')
        recovered = []
        still_failed = []
        
        for fail in failures:
            df, err = fetch_batch(fail['rics'], chunk_start, chunk_end, retries=5)
            if df is not None:
                recovered.append(df)
                print(f'  Recovered: {fail["rics"][:3]}...')
            elif err:
                still_failed.append(fail)
            time.sleep(SLEEP_BETWEEN * 2)
        
        if recovered:
            rec_raw = pd.concat(recovered, axis=0).dropna(how='all')
            rec_raw = rec_raw[~rec_raw.index.duplicated(keep='first')]
            
            if isinstance(rec_raw.columns, pd.MultiIndex):
                rec_long = rec_raw.stack(level=0).reset_index()
                rec_long = rec_long.rename(columns={'level_1': 'Instrument'})
                rec_long = rec_long.rename(columns={
                    'TRDPRC_1': 'Price',
                    'ACVOL_UNS': 'Volume'
                })
                rec_long = rec_long[['Date', 'Instrument', 'Price', 'Volume']]
            else:
                rec_long = rec_raw.reset_index()
                rec_long = rec_long.rename(columns={
                    'TRDPRC_1': 'Price',
                    'ACVOL_UNS': 'Volume'
                })
            
            rec_long['Price'] = pd.to_numeric(rec_long['Price'], errors='coerce')
            rec_long['Volume'] = pd.to_numeric(rec_long['Volume'], errors='coerce')
            rec_long = rec_long.dropna(subset=['Price', 'Volume'], how='all')
            
            chunk_file = f'{CHUNK_PREFIX}_{chunk_idx}.csv'
            existing = pd.read_csv(chunk_file, parse_dates=['Date'])
            combined = pd.concat([existing, rec_long], ignore_index=True)
            combined = combined.drop_duplicates(subset=['Date', 'Instrument'], keep='first')
            combined.to_csv(chunk_file, index=False)
            print(f'  +{len(rec_long):,} rows recovered')
        
        if still_failed:
            with open(ff, 'w') as f:
                json.dump(still_failed, f, indent=2)
            print(f'  {len(still_failed)} still failed')
        else:
            os.remove(ff)
            print(f'  All recovered!')
    
    print('\nRe-run Combine + Rebuild cells above.')

No failed batches to retry!
